In [ ]:
"""
04_random_forest.py
Amazon E-Commerce Sales - Profit Prediction
Model: Random Forest Regressor
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.ensemble           import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing      import RobustScaler
from sklearn.pipeline           import Pipeline
from sklearn.metrics            import (mean_absolute_error, mean_squared_error,
                                        r2_score, mean_absolute_percentage_error)

INPUT_PATH = "data/feature_engineered.csv"
MODEL_DIR  = "models"
PLOT_DIR   = "outputs/random_forest"
RESULT_DIR = "outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PLOT_DIR,  exist_ok=True)

RANDOM_STATE = 42
TARGET       = 'profit'

# Helper: Save figure
def save_fig(name):
    path = os.path.join(PLOT_DIR, f"{name}.png")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved plot: {path}")

# Helper: Print Metrics
def print_metrics(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"\n  Model: {name}")
    print(f"     MAE    : Rs. {mae:,.2f}")
    print(f"     RMSE   : Rs. {rmse:,.2f}")
    print(f"     R2     : {r2:.4f}")
    print(f"     MAPE   : {mape:.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE%': mape}

# 1. Load Data
print("="*60)
print(" LOADING FEATURE-ENGINEERED DATA")
print("="*60)
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"  Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# 2. Prepare Features
print("\n" + "="*60)
print(" PREPARING FEATURES")
print("="*60)

DROP_COLS = [
    TARGET, 'revenue', 'log_revenue', 'log_profit',
    'order_id', 'asin', 'sku', 'style',
    'ship_postal_code', 'promotion_ids', 'promotion-ids',
    'date', 'Date'
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

numeric_df   = df.select_dtypes(include='number')
feature_cols = [c for c in numeric_df.columns if c not in DROP_COLS]

X = df[feature_cols].copy()
y = df[TARGET].copy()

mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]
print(f"  Features: {len(feature_cols)}  |  Rows: {len(X):,}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print(f"  Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")

# 3. Baseline Random Forest
print("\n" + "="*60)
print(" BASELINE RANDOM FOREST")
print("="*60)

rf_base = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_base.fit(X_train, y_train)
y_pred_base = np.maximum(rf_base.predict(X_test), 0)
base_metrics = print_metrics("Random Forest (Baseline)", y_test, y_pred_base)

# 4. Hyperparameter Tuning
print("\n" + "="*60)
print(" HYPERPARAMETER TUNING (GridSearchCV 3-fold)")
print("="*60)

X_train_sub = X_train.sample(n=min(10000, len(X_train)), random_state=RANDOM_STATE)
y_train_sub = y_train.loc[X_train_sub.index]

param_grid = {
    'n_estimators'    : [100, 200],
    'max_depth'       : [15, 25],
    'min_samples_leaf': [2, 4],
    'max_features'    : ['sqrt'],
}

rf_gs = GridSearchCV(
    RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
    param_grid,
    cv=3,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)
rf_gs.fit(X_train_sub, y_train_sub)
print(f"\n  Best params: {rf_gs.best_params_}")
print(f"  Best CV R2 (subsampled) : {rf_gs.best_score_:.4f}")

best_rf = RandomForestRegressor(**rf_gs.best_params_, n_jobs=-1, random_state=RANDOM_STATE)
best_rf.fit(X_train, y_train)

y_pred_rf = np.maximum(best_rf.predict(X_test), 0)
tuned_metrics = print_metrics("Random Forest (Tuned)", y_test, y_pred_rf)
joblib.dump(best_rf, os.path.join(MODEL_DIR, 'random_forest_tuned.pkl'))
print(f"\n  Saved to: {MODEL_DIR}/random_forest_tuned.pkl")

# 5. Gradient Boosting
print("\n" + "="*60)
print(" GRADIENT BOOSTING REGRESSOR (comparison)")
print("="*60)

gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    random_state=RANDOM_STATE
)
gb.fit(X_train, y_train)
y_pred_gb = np.maximum(gb.predict(X_test), 0)
gb_metrics = print_metrics("Gradient Boosting", y_test, y_pred_gb)
joblib.dump(gb, os.path.join(MODEL_DIR, 'gradient_boosting.pkl'))

# 6. Cross-Validation
print("\n" + "="*60)
print(" 3-FOLD CROSS VALIDATION (Tuned RF)")
print("="*60)

X_sub = X.sample(n=min(30000, len(X)), random_state=RANDOM_STATE)
y_sub = y.loc[X_sub.index]

kf     = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_r2  = cross_val_score(best_rf, X_sub, y_sub, cv=kf, scoring='r2')
cv_mae = -cross_val_score(best_rf, X_sub, y_sub, cv=kf, scoring='neg_mean_absolute_error')
print(f"  CV R2  : {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")
print(f"  CV MAE : Rs. {cv_mae.mean():,.2f} +/- Rs. {cv_mae.std():,.2f}")

# 7. Feature Importance
print("\n" + "="*60)
print(" FEATURE IMPORTANCE")
print("="*60)

fi_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': best_rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(fi_df.head(20).to_string(index=False))
fi_df.to_csv(os.path.join(RESULT_DIR, "feature_importance_rf.csv"), index=False)

top20 = fi_df.head(20)
plt.figure(figsize=(10, 8))
cmap = plt.colormaps.get_cmap('Blues_r')
colours = [cmap(i / len(top20)) for i in range(len(top20))]
plt.barh(top20['Feature'], top20['Importance'], color=colours, edgecolor='white')
plt.gca().invert_yaxis()
plt.title("Top 20 Feature Importances - Random Forest", fontsize=13)
plt.xlabel("Importance Score")
plt.grid(axis='x', alpha=0.3)
save_fig("feature_importance")

# 8. Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Actual vs Predicted Profit", fontsize=14)

for ax, y_pred, title, colour in [
    (axes[0], y_pred_base, "Baseline RF", 'steelblue'),
    (axes[1], y_pred_rf,   "Tuned RF",    'darkorange')
]:
    ax.scatter(y_test, y_pred, alpha=0.35, color=colour, s=8)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect Fit')
    ax.set_xlabel("Actual Profit (INR)")
    ax.set_ylabel("Predicted Profit (INR)")
    ax.set_title(title); ax.legend()

save_fig("actual_vs_predicted")

# 9. Learning Curve
print("\n" + "="*60)
print(" LEARNING CURVE")
print("="*60)

from sklearn.model_selection import learning_curve

train_sizes, train_scores, test_scores = learning_curve(
    best_rf, X_sub, y_sub, cv=3,
    train_sizes=np.linspace(0.2, 1.0, 5),
    scoring='r2', n_jobs=-1
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue',  label='Train R2')
plt.plot(train_sizes, test_scores.mean(axis=1),  'o-', color='darkorange', label='CV R2')
plt.fill_between(train_sizes,
                 train_scores.mean(axis=1) - train_scores.std(axis=1),
                 train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color='steelblue')
plt.fill_between(train_sizes,
                 test_scores.mean(axis=1) - test_scores.std(axis=1),
                 test_scores.mean(axis=1) + test_scores.std(axis=1), alpha=0.15, color='darkorange')
plt.title("Learning Curve - Random Forest", fontsize=13)
plt.xlabel("Training Set Size"); plt.ylabel("R2 Score")
plt.legend(); plt.grid(alpha=0.3)
save_fig("learning_curve")

# 10. Model Comparison
results = pd.DataFrame([base_metrics, tuned_metrics, gb_metrics])
results.to_csv(os.path.join(RESULT_DIR, "random_forest_results.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Tree-Based Model Comparison", fontsize=14)

for ax, col, colour, label in [
    (axes[0], 'R2',   ['steelblue','darkorange','mediumseagreen'], "R2 Score"),
    (axes[1], 'RMSE', ['steelblue','darkorange','mediumseagreen'], "RMSE (INR)")
]:
    ax.bar(results['model'], results[col], color=colour, edgecolor='white', alpha=0.85)
    ax.set_ylabel(label); ax.set_title(f"{label} Comparison")
    ax.set_xticklabels(results['model'], rotation=15, ha='right')
    ax.grid(axis='y', alpha=0.3)

save_fig("rf_model_comparison")

print("\n" + "="*60)
print(" RANDOM FOREST COMPLETE")
print("="*60)
print(results.to_string(index=False))


 LOADING FEATURE-ENGINEERED DATA
  Loaded: 5,000 rows x 62 columns

 PREPARING FEATURES
  Features: 29  |  Rows: 5,000
  Train: 4,000 | Test: 1,000

 BASELINE RANDOM FOREST

  Model: Random Forest (Baseline)
     MAE    : Rs. 6.88
     RMSE   : Rs. 75.91
     R2     : 0.9493
     MAPE   : 0.73%

 HYPERPARAMETER TUNING (GridSearchCV 3-fold)
Fitting 3 folds for each of 8 candidates, totalling 24 fits
